# Low-Rank Approximation Error

Chapter 8: Singular Value Decomposition (SVD) · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare Frobenius error of rank-k truncations from singular values.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-8/low-rank-error/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's default singular values
sigma = torch.tensor([3.2, 1.4, 0.6], dtype=torch.float64)

# ── Build a 3x3 matrix with exactly these singular values ───────────────────
U, _ = torch.linalg.qr(torch.randn(3, 3, dtype=torch.float64))
V, _ = torch.linalg.qr(torch.randn(3, 3, dtype=torch.float64))
A = U @ torch.diag(sigma) @ V.T
Uf, Sf, Vhf = torch.linalg.svd(A)


def truncate(k):
    """Best rank-k approximation: keep the k largest singular triplets."""
    return Uf[:, :k] @ torch.diag(Sf[:k]) @ Vhf[:k]


# ── Eckart-Young: the error is exactly the discarded singular values ────────
print(" k   ||A - A_k||_F   sqrt(sum of dropped s^2)   ||A - A_k||_2   next s")
for k in range(4):
    E = A - truncate(k)
    fro = torch.linalg.matrix_norm(E).item()
    spec = torch.linalg.matrix_norm(E, ord=2).item() if k < 3 else 0.0
    pred = sigma[k:].pow(2).sum().sqrt().item()
    nxt = sigma[k].item() if k < 3 else 0.0
    print(f"{k:2d}   {fro:13.6f}   {pred:24.6f}   {spec:13.6f}   {nxt:6.3f}")

# Share of "energy" kept by rank k
energy = sigma.pow(2).cumsum(0) / sigma.pow(2).sum()
print("\nenergy kept by rank 1, 2, 3:", [round(e, 4) for e in energy.tolist()])

 k   ||A - A_k||_F   sqrt(sum of dropped s^2)   ||A - A_k||_2   next s
 0        3.544009                   3.544009        3.200000    3.200
 1        1.523155                   1.523155        1.400000    1.400
 2        0.600000                   0.600000        0.600000    0.600
 3        0.000000                   0.000000        0.000000    0.000

energy kept by rank 1, 2, 3: [0.8153, 0.9713, 1.0]
